# n007 — Interactive Artifact Phylogeny

Interactive visualisation of the artifact inheritance graph for a **single experiment run**. Nodes are laid out along the x-axis by creation time; hovering over a node shows its name, content, novelty score, category, and creation time.

**REQUIRES:** `004_artifact_analysis.py`, `005_artifact_classification.py`, and `006_artifact_phylogeny.py` to be run first.

Requires the `%matplotlib widget` backend (Jupyter notebook / VS Code interactive window). No files are saved to disk — this notebook is for exploration only.

**Key controls:**
- `EXP_NAME_TO_PLOT` — which experiment condition to visualise
- `EXP_IDX_TO_PLOT` — which run index within that condition (0-based)
- `SCORE_THRESHOLD` — minimum LLM confidence score for an edge to be shown

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
from core.utils import ROOT
from core.utils.analysis_utils import get_exp_folders
from collections import defaultdict
from analysis_scripts.artifact_complexity import ExperimentArtifacts
import io
import contextlib
import mplcursors
import textwrap
import pickle as pkl
import numpy as np
import json
from tqdm import tqdm
import importlib
importlib.reload(importlib.import_module('analysis_scripts.plot_utils'))
from analysis_scripts.plot_utils import plot_params


plt.rcParams.update(plot_params)

## Configuration

Set these three variables before running the notebook:
- `EXP_NAME_TO_PLOT` — must be one of the names in `EXP_BASE_NAMES`
- `EXP_IDX_TO_PLOT` — run index (0 to `n_runs - 1`)
- `SCORE_THRESHOLD` — edges with LLM confidence below this are hidden (0 = show all, 1 = only certain edges)

In [ ]:
EXP_BASE_NAMES = [] # TO DEFINE

EXP_NAME_TO_PLOT = ''
EXP_IDX_TO_PLOT = 3
SCORE_THRESHOLD = 0.8


# Change this if you want other names in the plot
PLOT_NAMES = {exp_name: exp_name.replace("_", " ").title() for exp_name in EXP_BASE_NAMES}

EXP_PATH = ROOT / "logs"
SHOW_STD = False
NOVELTY_MODEL = 'claude-sonnet-4-5-20250929'
MODEL = 'claude-haiku-4-5'
experiment_order = list(PLOT_NAMES.keys())


experiments = {}
for base_name in EXP_BASE_NAMES:
    experiments[base_name] = get_exp_folders(log_path=EXP_PATH, 
                                             exp_name=base_name, 
                                             only_completed=True)

## Data Loading

Identical to n005 — loads `ExperimentArtifacts`, the LLM phylogeny JSON, novelty scores, and category labels for every run. Novelty and category are attached to each artifact dict in-place.

In [ ]:
def validate_model_phylogeny(phyl_dict):
    invalid_nodes = []
    for node, data in phyl_dict.items():
        if not isinstance(data, dict):
            invalid_nodes.append(node)
            continue
        if 'error' in data:
            invalid_nodes.append(node)
            continue
    
    if invalid_nodes:
        return False, invalid_nodes
    else:
        return True, invalid_nodes

In [ ]:
artifacts = defaultdict(list)

f = io.StringIO()

phylogeny = {}
experiment_novelties = defaultdict(list)

for base_name, exps in experiments.items():
    phylogeny[base_name] = []
    for exp in tqdm(exps, desc='Exp: ' + base_name):
        exp_path = EXP_PATH / exp
        with contextlib.redirect_stdout(f):
            art = ExperimentArtifacts(exp_path=exp_path, embedding_dimensions=512, save_path=exp_path / "artifact_analysis")
            art.load(force_recalc=False)
            art._load_raw_artifacts()

        try:
            model_phyl = json.load(open(exp_path / "artifact_analysis" / f"artifact_phylogeny_{MODEL}.json", "r"))
        except:
            print("No phylogeny found for", exp)
            continue
        valid, invalid_nodes = validate_model_phylogeny(model_phyl)
        if not valid:
            print("Invalid nodes found in phylogeny for", exp, ":", invalid_nodes)


        try:
            with open(exp_path / 'artifact_analysis' / f'novelties_{NOVELTY_MODEL}.pkl', 'rb') as nov_file:
                novelties = pkl.load(nov_file)
        except FileNotFoundError:
            print(f"Cannot load novelties from {NOVELTY_MODEL} of {exp} ")
            novelties = {}

        try:
            with open(exp_path / 'artifact_analysis' / 'artifact_categories.json', 'r') as cat_file:
                categories = json.load(cat_file)
        except FileNotFoundError:
            print(f"Cannot load artifact categories of {exp} ")
            categories = {}

        experiment_novelties[base_name].append(novelties)

        for art_id in art.all_artifacts:
            try:
                nov = novelties[art_id]
            except KeyError:
                nov = -1
            art.all_artifacts[art_id]['novelty'] = nov
            art.all_artifacts[art_id]['category'] = categories.get(str(art_id), 0)

        phylogeny[base_name].append({'hand': json.load(open(exp_path / "artifact_analysis" / "artifact_phylogeny_hand.json", "r")),
                            MODEL: model_phyl})

        artifacts[base_name].append(art)
    print()

## Graph Utilities

Three helpers for building and laying out the phylogeny graph:

- **`to_nx_digraph(parents_by_node, artifacts, score_key)`** — converts the phylogeny dict to a `nx.DiGraph` with `parent → child` edges. Attaches `creation_time`, `novelty`, and `category` as node attributes when `artifacts` is provided. (Same as in n005.)
- **`get_graph_with_minimum_score(G, score_threshold)`** — returns a copy of `G` keeping only edges whose `score` attribute meets the threshold. All nodes are retained even if they become isolated.
- **`compute_time_based_layout(G, vertical_spacing, jitter)`** — assigns 2D positions where x = creation time and nodes at the same timestep are spread evenly on the y-axis. Used as the default layout for the interactive plot.

In [ ]:
import networkx as nx

def to_nx_digraph(parents_by_node, artifacts=None, score_key="score"):
    """
    parents_by_node:
      - {node: {parent: score, ...}}
      - {node: [parent, parent, ...]}

    Returns:
      nx.DiGraph with edges parent -> node
      and edge attribute `score` (default = 1.0)
    """
    G = nx.DiGraph()

    # --- collect all nodes ---
    nodes = set()
    for child, parents in parents_by_node.items():
        try:
            child = int(child)
            nodes.add(child)
        except:
            continue

        if isinstance(parents, dict):
            for p in parents.keys():
                try:
                    nodes.add(int(p))
                except:
                    pass
        else:  # list / iterable
            for p in parents:
                try:
                    nodes.add(int(p))
                except:
                    pass

    G.add_nodes_from(nodes)

    # --- add node attributes ---
    if artifacts is not None:
        for n in G.nodes:
            art = artifacts.get(int(n), {})
            G.nodes[n]["creation_time"] = art.get("creation_time", None)
            G.nodes[n]["novelty"] = max(art.get("novelty", 0.0), 0.0)
            G.nodes[n]["category"] = max(int(art.get("category", 0)), 0)

    # --- add edges ---
    for child, parents in parents_by_node.items():
        try:
            child = int(child)
        except:
            continue

        if isinstance(parents, dict):
            # parent -> child with score
            for p, score in parents.items():
                try:
                    p = int(p)
                    G.add_edge(p, child, **{score_key: float(score)})
                except:
                    continue
        else:
            # parent -> child with default score = 1
            for p in parents:
                try:
                    p = int(p)
                    G.add_edge(p, child, **{score_key: 1.0})
                except:
                    continue

    return G

def get_graph_with_minimum_score(G: nx.DiGraph, score_threshold: float, score_key="score") -> nx.DiGraph:
    """
    Returns a subgraph of G containing only edges with score >= score_threshold.
    Nodes that become disconnected (no incoming or outgoing edges) are also removed.

    Parameters:
    - G: Input directed graph (nx.DiGraph)
    - score_threshold: Minimum score for edges to be included
    - score_key: Edge attribute key for the score (default = "score")

    Returns:
    - Subgraph of G with edges meeting the score threshold
    """
    # Create a new directed graph to hold the filtered edges and nodes
    H = nx.DiGraph()
    # Add all nodes from G to H
    H.add_nodes_from(G.nodes(data=True))

    # Add edges that meet the score threshold
    for u, v, data in G.edges(data=True):
        if data.get(score_key, 0) >= score_threshold:
            H.add_edge(u, v, **data)


    return H

def compute_time_based_layout(G: nx.DiGraph, vertical_spacing=1.0, jitter=0.0):
    """
    Compute node positions with x-axis as creation time.
    Nodes at the same time are spread vertically.
    
    Args:
        G: NetworkX DiGraph with 'creation_time' node attribute
        vertical_spacing: Vertical spacing between nodes at same time
        jitter: Random jitter amount to add to y positions (0 = no jitter)
        
    Returns:
        pos: Dictionary of {node: (x, y)} positions
    """
    # Group nodes by creation time
    time_groups = {}
    time = 1
    for node in G.nodes():
        time = G.nodes[node].get('creation_time', time) + 1 # DO NOT CHANGE THIS
        if time not in time_groups:
            time_groups[time] = []
        time_groups[time].append(node)
    
    # Sort times
    sorted_times = sorted(time_groups.keys())
    
    # Assign positions
    pos = {}
    for time in sorted_times:
        nodes_at_time = time_groups[time]
        n_nodes = len(nodes_at_time)
        
        # Center nodes vertically
        for i, node in enumerate(nodes_at_time):
            y = (i - (n_nodes - 1) / 2) * vertical_spacing
            # Add random jitter
            if jitter > 0:
                y += np.random.uniform(-jitter, jitter)
            pos[node] = (time, y)
    
    return pos

## Neighbourhood Analysis

Two helpers for drilling into the local structure around specific nodes of interest:

- **`get_k_hop_neighborhood(G, target_nodes, k_hops)`** — returns two dicts (`ancestors`, `descendants`) mapping each target node to the set of nodes reachable within `k_hops` steps upstream or downstream.
- **`print_neighborhood_analysis(G, artifacts_dict, target_nodes, k_hops=3)`** — pretty-prints a structured report for each target node showing its artifact metadata, the top-5 ancestors and descendants by novelty score, its direct parents/children, and (when multiple targets are given) common ancestors and descendants across all targets.

In [ ]:
def get_k_hop_neighborhood(G: nx.DiGraph, target_nodes: list, k_hops: int):
    """
    Get k-hop ancestors and descendants for target nodes.
    
    Returns:
        ancestors, descendants: Two dictionaries mapping each target to sets of nodes
    """
    ancestors = {target: set() for target in target_nodes}
    descendants = {target: set() for target in target_nodes}
    
    for target in target_nodes:
        if target not in G:
            continue
            
        # Get ancestors (predecessors)
        for k in range(1, k_hops + 1):
            if k == 1:
                current_level = {target}
            else:
                current_level = new_level.copy()
            
            new_level = set()
            for node in current_level:
                preds = set(G.predecessors(node))
                new_level.update(preds)
                ancestors[target].update(preds)
        
        # Get descendants (successors)
        for k in range(1, k_hops + 1):
            if k == 1:
                current_level = {target}
            else:
                current_level = new_level.copy()
            
            new_level = set()
            for node in current_level:
                succs = set(G.successors(node))
                new_level.update(succs)
                descendants[target].update(succs)
    
    return ancestors, descendants

def print_neighborhood_analysis(G: nx.DiGraph, artifacts_dict: dict, 
                                target_nodes: list, k_hops=3):
    """
    Print detailed analysis of the neighborhood around target nodes.
    """
    print("=" * 80)
    print(f"NEIGHBORHOOD ANALYSIS FOR NODES: {target_nodes}")
    print("=" * 80)
    
    ancestors, descendants = get_k_hop_neighborhood(G, target_nodes, k_hops)
    
    for target in target_nodes:
        if target not in G:
            print(f"\n⚠️  Node {target} not found in graph")
            continue
            
        art = artifacts_dict.get(target, {})
        print(f"\n{'─' * 80}")
        print(f"TARGET NODE: {target}")
        print(f"  Name: {art.get('name', 'N/A')}")
        print(f"  Creator Tag: {art.get('creator_tag', 'N/A')}")
        print(f"  Time: {art.get('creation_time', 'N/A')}")
        print(f"  Novelty: {art.get('novelty', 0):.2f}")
        print(f"  Category: {art.get('category', 0)}")
        print(f"  Payload: {art.get('payload', 'N/A')}")
        
        # Ancestors (parents up to k hops)
        anc = ancestors[target]
        print(f"\n  {k_hops}-HOP ANCESTORS: {len(anc)} nodes")
        if anc:
            anc_sorted = sorted(anc, key=lambda n: artifacts_dict.get(n, {}).get('novelty', 0), reverse=True)
            print(f"    Top 5 by novelty:")
            for node in anc_sorted[:5]:
                a = artifacts_dict.get(node, {})
                print(f"      • Node {node}: {a.get('name', 'N/A')} (novelty={a.get('novelty', 0):.2f}, time={a.get('creation_time', 'N/A')})")
        
        # Descendants (children up to k hops)
        desc = descendants[target]
        print(f"\n  {k_hops}-HOP DESCENDANTS: {len(desc)} nodes")
        if desc:
            desc_sorted = sorted(desc, key=lambda n: artifacts_dict.get(n, {}).get('novelty', 0), reverse=True)
            print(f"    Top 5 by novelty:")
            for node in desc_sorted[:5]:
                d = artifacts_dict.get(node, {})
                print(f"      • Node {node}: {d.get('name', 'N/A')} (novelty={d.get('novelty', 0):.2f}, time={d.get('creation_time', 'N/A')})")
        
        # Direct connections
        direct_parents = list(G.predecessors(target))
        direct_children = list(G.successors(target))
        print(f"\n  DIRECT CONNECTIONS:")
        print(f"    Parents: {direct_parents}")
        print(f"    Children: {direct_children}")
    
    # Overlap analysis
    if len(target_nodes) > 1:
        print(f"\n{'─' * 80}")
        print("OVERLAP ANALYSIS:")
        all_ancestors = set.union(*[ancestors[t] for t in target_nodes if t in G])
        all_descendants = set.union(*[descendants[t] for t in target_nodes if t in G])
        
        # Common ancestors
        common_anc = set.intersection(*[ancestors[t] for t in target_nodes if t in G]) if all([t in G for t in target_nodes]) else set()
        if common_anc:
            print(f"  Common ancestors ({len(common_anc)}): {sorted(common_anc)}")
        
        # Common descendants
        common_desc = set.intersection(*[descendants[t] for t in target_nodes if t in G]) if all([t in G for t in target_nodes]) else set()
        if common_desc:
            print(f"  Common descendants ({len(common_desc)}): {sorted(common_desc)}")
    
    print("=" * 80 + "\n")

In [ ]:
phylogeny_to_plot = phylogeny[EXP_NAME_TO_PLOT][EXP_IDX_TO_PLOT][MODEL]
phylogeny_G = to_nx_digraph(phylogeny_to_plot, artifacts=artifacts[EXP_NAME_TO_PLOT][EXP_IDX_TO_PLOT].all_artifacts)
plot_G = get_graph_with_minimum_score(phylogeny_G, score_key="score", score_threshold=SCORE_THRESHOLD)

## Interactive Plot

**`plot_interactive_graph(G, artifacts_dict, ...)`** renders the filtered graph using `%matplotlib widget` so nodes are hoverable.

Node appearance:
- **Colour** — novelty score (viridis colormap; 0 = no novelty)
- **Size** — number of direct children (out-degree); larger = more influential artifact

Hover tooltip (via `mplcursors`) shows: artifact index, name, creation time, novelty score, category, and a truncated content preview (≤ 200 chars).

Layout options:
- `use_time_layout=True` *(default)* — time-aligned x-axis; set `log_scale=True` for a log x-axis when artifacts cluster near the start
- `use_time_layout=False` — spring layout (useful when creation times are unavailable or highly uniform)

In [ ]:
def plot_interactive_graph(G: nx.DiGraph, artifacts_dict: dict, figsize=(14, 8), 
                          use_time_layout=True, vertical_spacing=1.5, log_scale=False):
    """
    Create an interactive visualization of the graph where hovering shows node info.

    Args:
        G: NetworkX DiGraph
        artifacts_dict: Dictionary mapping node IDs to artifact data (with 'name', 'payload', etc.)
        figsize: Figure size tuple
        use_time_layout: If True, use creation_time for x-axis, else use spring layout
        vertical_spacing: Vertical spacing between nodes at same time (if use_time_layout=True)
        log_scale: If True, use log scale for time axis

    Returns:
        fig, ax: Matplotlib figure and axes objects
    """
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_facecolor("#f7f7f7")

    # Compute layout
    if use_time_layout:
        pos = compute_time_based_layout(G, vertical_spacing=vertical_spacing)
    else:
        pos = nx.spring_layout(G, k=2, iterations=50, seed=42)

    # Draw edges with reduced visual clutter
    for u, v in G.edges():
        x1, y1 = pos[u]
        x2, y2 = pos[v]
        ax.plot([x1, x2], [y1, y2], color='gray', linewidth=1.5, 
                alpha=0.3, zorder=1)

    # Get node attributes
    nodes_list = list(G.nodes())
    xs = [pos[n][0] for n in nodes_list]
    ys = [pos[n][1] for n in nodes_list]

    # Color by novelty if available
    colors = []
    sizes = []
    for n in nodes_list:
        novelty = G.nodes[n].get('novelty', 0)
        colors.append(novelty)
        # Size by number of children
        n_children = len(list(G.successors(n)))
        sizes.append(150 + n_children * 40)

    # Create scatter plot
    scatter = ax.scatter(xs, ys, c=colors, s=sizes,
                        cmap='viridis', edgecolors='white',
                        linewidths=2, zorder=3, alpha=0.9)

    # Add colorbar
    cbar = plt.colorbar(scatter, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Novelty', fontsize=10)

    # Add node labels (indices)
    for node, (x, y) in pos.items():
        ax.text(x, y, str(node), fontsize=7, ha='center', va='center',
               bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                        edgecolor='none', alpha=0.8), zorder=4)

    # Setup hover tooltips
    cursor = mplcursors.cursor(scatter, hover=True)

    @cursor.connect("add")
    def on_add(sel):
        idx = sel.index
        node_id = nodes_list[idx]

        # Get artifact data
        art = artifacts_dict.get(node_id, {})
        name = art.get('name', f'Node {node_id}')
        novelty = art.get('novelty', 0)
        category = art.get('category', 0)
        creation_time = art.get('creation_time', 'N/A')
        payload = art.get('payload', 'No content')

        # Limit payload length for display
        if len(payload) > 200:
            payload = payload[:200] + '...'

        # Create tooltip text
        tooltip_text = f"Index: {node_id}\n"
        tooltip_text += f"Name: {name}\n"
        tooltip_text += f"Time: {creation_time}\n"
        tooltip_text += f"Novelty: {novelty:.2f}\n"
        tooltip_text += f"Category: {category}\n"
        tooltip_text += f"\nContent:\n{textwrap.fill(payload, width=50)}"

        sel.annotation.set_text(tooltip_text)
        sel.annotation.set_fontsize(9)
        sel.annotation.get_bbox_patch().set(facecolor='white', alpha=0.95,
                                            edgecolor='black', linewidth=1)
        sel.annotation.set_ha("left")
        sel.annotation.set_va("center")

    # Axis formatting
    if use_time_layout:
        ax.set_xlabel("Creation Time" + (" (log scale)" if log_scale else ""), fontsize=11)
        ax.set_yticks([])
        ax.spines['left'].set_visible(False)
        if log_scale:
            ax.set_xscale('log')
        ax.grid(True, axis='x', alpha=0.3, which='both')
    else:
        ax.axis('off')
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    ax.set_title("Interactive Graph - Hover over nodes to see details", fontsize=13, pad=15)
    plt.tight_layout()

    return fig, ax

In [ ]:
fig1, ax1 = plot_interactive_graph(
    G=plot_G,
    artifacts_dict=artifacts[EXP_NAME_TO_PLOT][EXP_IDX_TO_PLOT].all_artifacts,
    figsize=(16, 10),
    use_time_layout=True,  # Use time-based x-axis
    vertical_spacing=1.5,  # Adjust vertical spacing for nodes at same time
    log_scale=True  # Set to True for logarithmic time scale
)
plt.show()

## Neighbourhood Inspection

Set `target_node_indices` to the artifact IDs you want to inspect (readable from the interactive graph above). Runs `print_neighborhood_analysis` on `plot_G` (the score-filtered graph for the selected run).

In [ ]:
target_node_indices = [5, 21, 16, 26, 53, 90]  # Change these to the nodes you're interested in

print_neighborhood_analysis(artifacts_dict=artifacts[EXP_NAME_TO_PLOT][EXP_IDX_TO_PLOT].all_artifacts,
    target_nodes=target_node_indices, G=plot_G)